# Logistic Regression

## Import Dependencies

In [4]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## Prepare Data

In [6]:
bc = datasets.load_breast_cancer()
X, y = bc.data, bc.target

n_samples, n_features = X.shape
print(f"Number of samples: {n_samples}, Number of features: {n_features}")

Number of samples: 569, Number of features: 30


In [7]:
# split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# scale features - 0 mean, unit variance - recommended
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [9]:
# convert to tensors
X_train = torch.from_numpy(X_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
y_test = torch.from_numpy(y_test.astype(np.float32))

In [10]:
# reshape the tensors
y_train = y_train.view(y_train.shape[0], 1)
y_test = y_test.view(y_test.shape[0], 1)

## Model

$$
\mathbf{y} = W * \mathbf{x} + b
$$
Activation: sigmoid

In [11]:
class LogisticRegression(nn.Module):
    def __init__(self, n_input_features):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(n_input_features, 1)

    def forward(self, x):
        y_pred = torch.sigmoid(self.linear(x))
        return y_pred

model = LogisticRegression(n_features)

## Loss & Optimizer

In [12]:
learning_rate = 0.01

criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

## Training Loop

In [13]:
num_epochs = 100

for epoch in range(num_epochs):
    # forward pass & loss
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    
    # backward pass
    loss.backward()

    # updates
    optimizer.step()

    # empty gradients
    optimizer.zero_grad()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch}: loss = {loss.item()}")

Epoch 9: loss = 0.7083525061607361
Epoch 19: loss = 0.5773674845695496
Epoch 29: loss = 0.49418869614601135
Epoch 39: loss = 0.43698543310165405
Epoch 49: loss = 0.39515864849090576
Epoch 59: loss = 0.36312252283096313
Epoch 69: loss = 0.33769679069519043
Epoch 79: loss = 0.316950261592865
Epoch 89: loss = 0.2996455132961273
Epoch 99: loss = 0.2849535346031189


## Evaluation

In [20]:
# remember we don't need gradients anymore after training
with torch.no_grad():
    y_pred_test = model(X_test)
    y_pred_cls = y_pred_test.round()
    acc = y_pred_cls.eq(y_test).sum() / float(y_test.shape[0])
    print(f"Accuracy = {acc}")

Accuracy = 0.9649122953414917
